# DSPy-Guided Synthetic Image Generation with Pruna Optimization

## Overview

This cookbook demonstrates how to build a closed-loop synthetic image generation pipeline that combines:

1. Inference Optimization with Pruna
Quantization, compilation, and runtime optimization for faster image generation.

2. Prompt Refinement with DSPy
Structured prompt generation and iterative improvement using programmable LLM pipelines.

3. Vision-Language Evaluation
Automated image quality assessment using a SmolVLM-based evaluator.

4.Ranking and Dataset Curation
Top-K filtering to retain only the highest-quality synthetic samples.

The final pipeline produces a curated synthetic dataset by iteratively generating, evaluating and refining image outputs.


## 1. Installation & Setup

### Prerequisites

This tutorial uses:

Pruna for inference optimization and model compression, DSPy for structured prompt orchestration, Transformers for VLM-based evaluation
, Datasets for synthetic dataset construction and Pillow for image processing utilities

In [ ]:
!pip install pruna dspy-ai transformers pillow datasets accelerate

In [ ]:
import os
import json
import logging
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

# DSPy
import dspy
from dspy import settings

# Pruna
from pruna import smash, SmashConfig, PrunaModel

# Hugging Face
import datasets
from huggingface_hub import notebook_login
from diffusers import Flux2Pipeline

logger = logging.getLogger(__name__)

## Authentication

Authenticate with Hugging Face to download gated models and upload generated datasets.

In [ ]:
notebook_login()

### Device & Cache Setup

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

In [ ]:
OUTPUT_DIR = Path("./flux_dataset_output")
OUTPUT_DIR.mkdir(exist_ok=True)

IMAGES_DIR = OUTPUT_DIR / "images"
IMAGES_DIR.mkdir(exist_ok=True)

METADATA_PATH = OUTPUT_DIR / "metadata.json"

## Part A: Optimize FLUX.2-klein-4B with Pruna

Before generating synthetic images, we first optimize the base diffusion pipeline using Pruna.

Our optimization strategy focuses on:

INT8 transformer quantization to reduce memory usage
Torch compilation to improve kernel execution efficiency
preserving generation quality for downstream dataset curation

Rather than aggressively minimizing VRAM, the configuration below prioritizes a balance between:

inference speed
memory efficiency
visual fidelity

### A1: Load Base Model

In [ ]:
logger.info("Loading FLUX.2-klein-4B...")

pipe = Flux2Pipeline.from_pretrained(
    "black-forest-labs/FLUX.2-klein-4B"
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

logger.info("✓ Base model loaded")

### A2: Define Pruna SmashConfig

In [ ]:
logger.info("Creating SmashConfig...")

smash_config = SmashConfig({

    # ------------------------------------------------------------------
    # INT8 Quantization
    # ------------------------------------------------------------------
    "diffusers_int8": {
        "diffusers_int8_weight_bits": 8,

        # FP4 quantization provides a good balance
        # between compression and image quality.
        "diffusers_int8_quant_type": "fp4",

        # Double quantization is unnecessary
        # for a 4B-scale diffusion model.
        "diffusers_int8_double_quant": False,

        # Keep weights on GPU for lower latency.
        "diffusers_int8_enable_fp32_cpu_offload": False,

        # Quantize transformer-heavy components only.
        "diffusers_int8_target_modules": {
            "include": [
                "*transformer*",
                "*attn*"
            ],
            "exclude": [
                "*norm*",
                "*embed*",
                "*pe_*"
            ]
        },
    },

    # ------------------------------------------------------------------
    # Torch Compile
    # ------------------------------------------------------------------
    "torch_compile": {
        "torch_compile_mode": "reduce-overhead",
        "torch_compile_backend": "inductor",

        # Avoid overly rigid graph specialization.
        "torch_compile_fullgraph": False,

        # Static shapes improve compilation stability.
        "torch_compile_dynamic": False,

        # Improves portability across environments.
        "torch_compile_make_portable": True,

        "torch_compile_target": "module_list",
    },
})

logger.info("✓ SmashConfig created")

### A3: Apply Pruna Smash

In [ ]:
logger.info("Applying Pruna optimizations...")

flux = smash(
    pipe,
    smash_config
)

logger.info("✓ Model optimization complete")

### A4: Save Optimized Model

In [ ]:
SMASHED_MODEL_PATH = OUTPUT_DIR / "flux_klein_4b_smashed"

logger.info(f"Saving optimized model to {SMASHED_MODEL_PATH}")

flux.save_pretrained(
    str(SMASHED_MODEL_PATH)
)

logger.info("✓ Optimized model saved")

## Part B: Configure VLM Grader (SmolVLM + DSPy)

To automatically curate generated images, we use a lightweight Vision-Language Model (VLM) as an evaluator.

The evaluator is responsible for:

scoring image quality
validating prompt alignment
filtering low-quality generations
enabling closed-loop dataset refinement

For this tutorial, we use SmolVLM-Instruct as the grading backend because it is lightweight enough for repeated inference while still capable of multimodal reasoning.

### B1: Initialize SmolVLM via DSPy

In [ ]:
logger.info("Initializing SmolVLM evaluator...")

vlm = dspy.HFModel(
    model="HuggingFaceTB/SmolVLM-Instruct",

    # Indicates that the backend accepts image inputs.
    hf_type="vision_model",

    # Deterministic decoding improves grading consistency.
    temperature=0.3,
    top_p=1.0,

    # Evaluation responses are intentionally short.
    max_tokens=256,
)

# Configure DSPy globally.
settings.configure(
    lm=vlm
)

logger.info("✓ DSPy configured with SmolVLM")

## Part C: Define Grading Schema

Rather than generating free-form critiques, we use a structured grading schema to make evaluation outputs:

machine-readable
composable
rankable
suitable for dataset filtering

DSPy signatures provide a declarative interface for defining multimodal evaluation contracts between the image generator and the VLM evaluator.

### C1: Image Grade Signature

In [ ]:
class ImageGrade(dspy.Signature):
    """
    Structured multimodal grading schema
    for synthetic image evaluation.
    """

    # ------------------------------------------------------------------
    # Inputs
    # ------------------------------------------------------------------

    prompt: str = dspy.InputField(
        desc="The original text prompt used for image generation"
    )

    image: object = dspy.InputField(
        desc="Generated PIL image to evaluate"
    )

    # ------------------------------------------------------------------
    # Evaluation Scores
    # ------------------------------------------------------------------

    prompt_adherence: float = dspy.OutputField(
        desc="0-10 score measuring semantic alignment between prompt and image"
    )

    aesthetic_quality: float = dspy.OutputField(
        desc="0-10 score for composition, clarity, lighting, and visual quality"
    )

    text_correctness: float = dspy.OutputField(
        desc="0-10 score evaluating rendered text accuracy if text is present"
    )

    brand_alignment: float = dspy.OutputField(
        desc="0-10 score for stylistic consistency and professional appearance"
    )

### C2: VLM Grader Module

In [ ]:
class VLMGrader(dspy.Module):
    """
    DSPy module for multimodal image evaluation.
    """

    def __init__(self):
        super().__init__()

        self.predictor = dspy.Predict(
            ImageGrade
        )

    def forward(
        self,
        prompt: str,
        image: Image.Image
    ) -> Dict[str, float]:

        prediction = self.predictor(
            prompt=prompt,
            image=image
        )

        return {
            "prompt_adherence": float(prediction.prompt_adherence),
            "aesthetic_quality": float(prediction.aesthetic_quality),
            "text_correctness": float(prediction.text_correctness),
            "brand_alignment": float(prediction.brand_alignment),
        }

### C3: Composite Scoring Metric

In [ ]:
class ScoringWeights:
    """
    Weighting strategy for dataset curation.

    Prompt adherence is prioritized because
    semantic correctness is more important
    than purely aesthetic quality.
    """

    PROMPT_ADHERENCE = 0.40
    AESTHETIC_QUALITY = 0.30
    TEXT_CORRECTNESS = 0.20
    BRAND_ALIGNMENT = 0.10


In [ ]:
def compute_composite_score(
    scores: Dict[str, float]
) -> float:
    """
    Compute weighted ranking score for image selection.
    """

    return (
        ScoringWeights.PROMPT_ADHERENCE
        * scores.get("prompt_adherence", 0.0)

        + ScoringWeights.AESTHETIC_QUALITY
        * scores.get("aesthetic_quality", 0.0)

        + ScoringWeights.TEXT_CORRECTNESS
        * scores.get("text_correctness", 0.0)

        + ScoringWeights.BRAND_ALIGNMENT
        * scores.get("brand_alignment", 0.0)
    )

## Part D: Dataset Generation Loop

With the optimized FLUX pipeline and VLM evaluator in place, we can now build the core dataset generation loop.

The workflow follows a simple closed-loop structure:

Subject
   ↓
Prompt Optimization
   ↓
Image Generation
   ↓
VLM Evaluation
   ↓
Composite Scoring
   ↓
Top-K Selection

For each subject:

A generation prompt is refined
Multiple candidate images are generated
Each image is evaluated using the VLM grader
Candidates are ranked using a composite score
The highest-quality samples are retained

### D1: Define Controlled Subjects

We use a small curated subject list to keep the dataset focused and reproducible.

In [ ]:
# These are your dataset subjects
# Keep this curated and small
SUBJECTS = [
    "Minimalist coffee brand poster with clean typography and steam",
    "Modern eco-friendly packaging design for luxury chocolate",
    "Contemporary tech startup office entrance with glass and wood",
    "Artisanal bakery storefront with warm lighting and fresh bread display",
    "Sustainable fashion brand lookbook with neutral tones and natural fabrics",
    "Boutique hotel lobby with marble and soft ambient lighting",
    "Organic skincare product flat lay with natural ingredients",
    "Furniture showroom displaying minimalist wooden pieces",
    "Plant-based restaurant menu photography with vibrant ingredients",
    "Luxury watch brand catalog shot with professional photography",
]

# For quick testing, use subset
SUBJECTS_TEST = SUBJECTS[:3]

logger.info(f"Using {len(SUBJECTS)} subjects for full run")
logger.info(f"Using {len(SUBJECTS_TEST)} subjects for test run")

D2. Prompt Optimization with DSPy

DSPy is used here to structure prompt refinement before image generation.

For this tutorial, we use a lightweight deterministic enrichment strategy. In production systems, this module could be extended with:

few-shot prompting
Chain-of-Thought refinement
iterative optimization loops

In [ ]:
class PromptOptimizer(dspy.Module):
    """
    Simple prompt enrichment module
    for image generation.
    """

    def forward(self, subject: str) -> str:

        return (
            f"{subject}, "
            "professional product photography, "
            "soft natural lighting, "
            "high detail, "
            "clean composition, "
            "4K"
        )


prompt_optimizer = PromptOptimizer()

D3. Generate and Evaluate Candidates

For each subject:

generate multiple candidate images
evaluate each image using the VLM grader
compute a composite ranking score

In [ ]:
def generate_candidates(
    subject: str,
    num_candidates: int = 4,
):
    """
    Generate and evaluate candidate images
    for a single subject.
    """

    optimized_prompt = prompt_optimizer(
        subject=subject
    )

    logger.info(f"Generating candidates for: {subject}")

    candidates = []

    for idx in range(num_candidates):

        image = flux(
            prompt=optimized_prompt,
            num_inference_steps=20,
            guidance_scale=3.5,
            height=768,
            width=768,
        ).images[0]

        scores = vlm_grader(
            prompt=optimized_prompt,
            image=image
        )

        scores["composite"] = compute_composite_score(
            scores
        )

        logger.info(
            f"Candidate {idx + 1}: "
            f"{scores['composite']:.2f}/10"
        )

        candidates.append(
            (image, scores, optimized_prompt)
        )

    return candidates

D4. Select the Top-K Candidates

After evaluation, candidates are ranked by composite score and filtered to retain only the strongest samples.

In [ ]:
def select_top_k(
    candidates,
    k: int = 2
):
    """
    Rank candidates by composite score
    and retain the top-k samples.
    """

    candidates.sort(
        key=lambda x: x[1]["composite"],
        reverse=True
    )

    return candidates[:k]

D5. Run the Dataset Generation Pipeline

We now combine generation, evaluation, ranking, and persistence into a single pipeline.

In [ ]:
def generate_dataset(
    subjects,
    candidates_per_subject: int = 4,
    keep_top_k: int = 2,
):

    dataset_records = []

    for subject_idx, subject in enumerate(subjects, start=1):

        candidates = generate_candidates(
            subject=subject,
            num_candidates=candidates_per_subject,
        )

        selected = select_top_k(
            candidates,
            k=keep_top_k,
        )

        for rank, (image, scores, prompt) in enumerate(selected, start=1):

            image_id = (
                f"subject_{subject_idx:02d}_rank_{rank}"
            )

            image_path = (
                IMAGES_DIR / f"{image_id}.png"
            )

            image.save(image_path)

            dataset_records.append({
                "image_id": image_id,
                "subject": subject,
                "prompt": prompt,
                "image_path": str(
                    image_path.relative_to(OUTPUT_DIR)
                ),
                "scores": scores,
            })

    return dataset_records

D6. Execute the Pipeline


dataset_records = generate_dataset(
    subjects=SUBJECTS,
    candidates_per_subject=4,
    keep_top_k=2,
)

E — Exporting the Dataset to Hugging Face Hub

At this stage, the pipeline has generated:

curated synthetic images
structured evaluation scores
optimized prompts
ranked candidate selections

We now package the results as a Hugging Face Dataset and publish them to the Hub.

In [ ]:
hf_records = []

for record in dataset_records:

    hf_records.append({
        "image_id": record["image_id"],
        "subject": record["subject"],
        "prompt": record["prompt"],

        # Use the standard HF image column name.
        "image": str(
            OUTPUT_DIR / record["image_path"]
        ),

        # Flatten evaluation metrics.
        "prompt_adherence": record["scores"]["prompt_adherence"],
        "aesthetic_quality": record["scores"]["aesthetic_quality"],
        "text_correctness": record["scores"]["text_correctness"],
        "brand_alignment": record["scores"]["brand_alignment"],
        "composite_score": record["scores"]["composite"],
    })

logger.info(
    f"Prepared {len(hf_records)} dataset records"
)

E2. Create the Hugging Face Dataset


In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_list(
    hf_records
)

hf_dataset

In [ ]:
from datasets import Image

hf_dataset = hf_dataset.cast_column(
    "image",
    Image()
)

hf_dataset

E5. Define the Dataset Repository and Push

Choose a repository name for the dataset.

In [ ]:
HF_DATASET_REPO = (
    "paragekbote/flux-synthetic-dataset"
)

hf_dataset.push_to_hub(
    HF_DATASET_REPO
)

## Conclusion

This cookbook demonstrated how to build a closed-loop synthetic image dataset pipeline by combining:

Pruna for optimized FLUX.2-klein-4B inference
DSPy for structured orchestration and evaluation workflows
SmolVLM for automated multimodal quality assessment
Top-K ranking for dataset curation and filtering

Rather than treating image generation as a single-pass process, the pipeline introduces an iterative generate → evaluate → rank workflow that enables higher-quality synthetic dataset construction with minimal manual review.

The final system supports:

optimized diffusion inference
structured VLM-based evaluation
reproducible candidate ranking
automated dataset packaging
direct publishing to the Hugging Face Hub

By combining model optimization, structured evaluation, and dataset curation into a unified workflow, the pipeline becomes significantly more scalable than manual synthetic data generation approaches.